# Projet Time Series — Monthly Armed Robberies in Boston

**Problème 1 : Monthly Armed Robberies in Boston**  
Ce notebook suit la structure demandée dans l'énoncé du projet : analyse exploratoire, préparation, stationnarité, transformations, identification ACF/PACF, modélisation, évaluation, diagnostic des résidus et conclusion.

**Objectif :** construire et comparer plusieurs modèles de prévision pour une série mensuelle représentant le nombre de vols à main armée à Boston.

> Remarque : le notebook est volontairement pédagogique. Les cellules Markdown expliquent la logique et les cellules Python exécutent l'analyse.

## 0. Préparation de l'environnement

Nous importons les bibliothèques nécessaires :

- `pandas` pour manipuler les données ;
- `numpy` pour les calculs numériques ;
- `matplotlib` pour les visualisations ;
- `statsmodels` pour les tests statistiques et les modèles de séries temporelles ;
- `scipy` pour certains tests complémentaires.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from scipy.stats import shapiro

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

## 1. Chargement et présentation de la série

La série étudiée contient deux colonnes :

- `Months` : date mensuelle de l'observation ;
- `Robberies` : nombre de vols à main armée enregistrés pour le mois correspondant.

Comme il s'agit d'une série temporelle mensuelle, nous transformons la colonne `Months` en index temporel avec une fréquence mensuelle.

In [ ]:
# Le fichier doit se trouver dans le même dossier que le notebook ou dans /mnt/data lors de l'exécution sur ChatGPT.
possible_paths = [
    Path("Robberies.csv"),
    Path("../data/Robberies.csv"),
    Path("/mnt/data/Robberies.csv")
]

csv_path = None
for path in possible_paths:
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("Placez le fichier Robberies.csv dans le même dossier que ce notebook ou dans un dossier data/.")

raw_df = pd.read_csv(csv_path)
raw_df.head()

,collection_date,crime_guns_recovered,guns_recovered_safeguard,buyback_guns_recovered
0,2014-08-20,2.0,3.0,1.0
1,2014-08-21,2.0,0.0,4.0
2,2014-08-22,0.0,0.0,2.0
3,2014-08-25,8.0,3.0,0.0
4,2014-08-26,9.0,0.0,0.0


In [ ]:
df = raw_df.copy()
df["Months"] = pd.to_datetime(df["Months"])
df = df.set_index("Months").sort_index()

# On force la fréquence mensuelle début de mois si possible.
df = df.asfreq("MS")

df.info()

In [ ]:
df.head(), df.tail(), df.shape

### Commentaire initial

Chaque observation correspond à un mois. La variable étudiée est le nombre de vols à main armée. Comme les valeurs sont des nombres d'événements, l'unité d'interprétation des erreurs sera aussi le **nombre de vols par mois**.

## 2. Analyse exploratoire

L'objectif de l'analyse exploratoire est de comprendre la dynamique générale de la série avant de modéliser.

Nous allons étudier :

- l'évolution temporelle ;
- les statistiques descriptives ;
- la tendance ;
- la variance ;
- la saisonnalité potentielle ;
- les valeurs atypiques éventuelles.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["Robberies"], marker="o", linewidth=1)
ax.set_title("Monthly Armed Robberies in Boston")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de vols")
plt.show()

### Statistiques descriptives

Soit une série temporelle $Y_1, Y_2, ..., Y_n$.

La moyenne empirique est définie par :

$$
\bar{Y} = \frac{1}{n}\sum_{t=1}^{n}Y_t
$$

La variance empirique mesure la dispersion autour de la moyenne :

$$
s^2 = \frac{1}{n-1}\sum_{t=1}^{n}(Y_t - \bar{Y})^2
$$

L'écart-type est la racine carrée de la variance :

$$
s = \sqrt{s^2}
$$

Dans notre contexte, ces statistiques permettent de décrire le niveau moyen mensuel des vols et leur variabilité.

In [ ]:
stats = pd.DataFrame({
    "Statistique": ["Nombre d'observations", "Moyenne", "Variance", "Écart-type", "Minimum", "Maximum"],
    "Valeur": [
        len(df),
        df["Robberies"].mean(),
        df["Robberies"].var(ddof=1),
        df["Robberies"].std(ddof=1),
        df["Robberies"].min(),
        df["Robberies"].max()
    ]
})
stats

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["Robberies"], bins=15, edgecolor="black")
ax.set_title("Distribution du nombre mensuel de vols")
ax.set_xlabel("Nombre de vols")
ax.set_ylabel("Fréquence")
plt.show()

In [ ]:
# Moyenne mobile pour visualiser la tendance locale
rolling_window = 12
rolling_mean = df["Robberies"].rolling(window=rolling_window).mean()
rolling_std = df["Robberies"].rolling(window=rolling_window).std()

fig, ax = plt.subplots()
ax.plot(df.index, df["Robberies"], label="Série originale", alpha=0.7)
ax.plot(rolling_mean.index, rolling_mean, label="Moyenne mobile 12 mois", linewidth=2)
ax.plot(rolling_std.index, rolling_std, label="Écart-type mobile 12 mois", linewidth=2)
ax.set_title("Tendance et variabilité locale")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de vols")
ax.legend()
plt.show()

### Interprétation graphique attendue

À partir du graphique, on doit discuter les points suivants :

- si la série augmente ou diminue dans le temps ;
- si la moyenne semble stable ou non ;
- si la variance semble constante ou croissante ;
- si un motif saisonnier se répète tous les 12 mois ;
- si certaines observations semblent atypiques.

Dans cette série, on s'attend généralement à observer une tendance croissante, ce qui peut indiquer une non-stationnarité.

## 3. Décomposition de la série

Une série temporelle peut être décomposée en trois composantes :

- **Tendance** : évolution globale de long terme ;
- **Saisonnalité** : variations périodiques régulières ;
- **Résidu** : partie non expliquée par la tendance et la saisonnalité.

Deux modèles sont généralement utilisés :

### Modèle additif

$$
Y_t = T_t + S_t + R_t
$$

Il est adapté lorsque l'amplitude des variations saisonnières reste globalement stable.

### Modèle multiplicatif

$$
Y_t = T_t \times S_t \times R_t
$$

Il est adapté lorsque l'amplitude saisonnière augmente avec le niveau de la série.

In [ ]:
decomp_add = seasonal_decompose(df["Robberies"], model="additive", period=12)
decomp_add.plot()
plt.suptitle("Décomposition additive", y=1.02)
plt.show()

In [ ]:
# La décomposition multiplicative exige des valeurs strictement positives.
decomp_mul = seasonal_decompose(df["Robberies"], model="multiplicative", period=12)
decomp_mul.plot()
plt.suptitle("Décomposition multiplicative", y=1.02)
plt.show()

### Interprétation de la décomposition

À commenter dans le rapport :

- **Tendance :** indique si le phénomène augmente ou diminue globalement ;
- **Saisonnalité :** montre si certains mois sont systématiquement plus élevés ou plus faibles ;
- **Résidu :** contient ce qui reste après extraction de la tendance et de la saisonnalité.

Si les résidus gardent une structure visible, cela signifie que la décomposition ne capture pas toute la dynamique.

## 4. Hypothèses initiales sur la dynamique

D'après les visualisations précédentes, on peut formuler les hypothèses suivantes :

1. La série semble probablement non stationnaire à cause d'une tendance croissante.
2. Une transformation ou une différenciation peut être nécessaire.
3. Une saisonnalité mensuelle est possible, mais doit être confirmée par la décomposition et les graphiques ACF/PACF.
4. Un modèle ARIMA peut être pertinent si la différenciation rend la série stationnaire.
5. Un modèle SARIMA peut être testé si la saisonnalité est significative.

## 5. Feature engineering

Nous créons quelques variables explicatives utiles pour comprendre la série :

- année ;
- mois ;
- trimestre ;
- retards temporels ;
- moyennes mobiles.

Ces variables ne sont pas toutes utilisées directement dans ARIMA, mais elles aident à mieux analyser la dynamique.

In [ ]:
features = df.copy()
features["year"] = features.index.year
features["month"] = features.index.month
features["quarter"] = features.index.quarter
features["lag_1"] = features["Robberies"].shift(1)
features["lag_12"] = features["Robberies"].shift(12)
features["rolling_mean_3"] = features["Robberies"].rolling(window=3).mean()
features["rolling_mean_12"] = features["Robberies"].rolling(window=12).mean()
features.head(15)

In [ ]:
monthly_profile = features.groupby("month")["Robberies"].agg(["mean", "std", "min", "max"])
monthly_profile

In [ ]:
fig, ax = plt.subplots()
monthly_profile["mean"].plot(kind="bar", ax=ax)
ax.set_title("Moyenne des vols par mois")
ax.set_xlabel("Mois")
ax.set_ylabel("Nombre moyen de vols")
plt.show()

## 6. Tests de stationnarité : ADF et KPSS

Une série est stationnaire si ses propriétés statistiques, comme la moyenne et la variance, restent stables dans le temps.

### Test ADF

Hypothèses :

$$
H_0 : \text{la série possède une racine unitaire, donc elle n'est pas stationnaire}
$$

$$
H_1 : \text{la série est stationnaire}
$$

Règle de décision : si la p-value est inférieure à 0.05, on rejette $H_0$.

### Test KPSS

Hypothèses :

$$
H_0 : \text{la série est stationnaire}
$$

$$
H_1 : \text{la série n'est pas stationnaire}
$$

Règle de décision : si la p-value est inférieure à 0.05, on rejette $H_0$.

Les deux tests sont complémentaires car leurs hypothèses nulles sont opposées.

In [ ]:
def stationarity_tests(series, name="Série"):
    series = pd.Series(series).dropna()

    adf_result = adfuller(series, autolag="AIC")
    kpss_result = kpss(series, regression="c", nlags="auto")

    results = pd.DataFrame({
        "Test": ["ADF", "KPSS"],
        "Statistique": [adf_result[0], kpss_result[0]],
        "p-value": [adf_result[1], kpss_result[1]],
        "Conclusion à 5%": [
            "Stationnaire" if adf_result[1] < 0.05 else "Non stationnaire",
            "Non stationnaire" if kpss_result[1] < 0.05 else "Stationnaire"
        ]
    })
    print(f"Résultats des tests de stationnarité — {name}")
    return results

stationarity_tests(df["Robberies"], "Série originale")

## 7. Transformations et différenciation

Pour stabiliser une série non stationnaire, on peut appliquer :

- une transformation logarithmique ;
- une transformation racine carrée ;
- une différenciation.

L'opérateur de retard $L$ est défini par :

$$
L Y_t = Y_{t-1}
$$

La différenciation d'ordre 1 est :

$$
\nabla Y_t = Y_t - Y_{t-1} = (1-L)Y_t
$$

La différenciation d'ordre $d$ est :

$$
\nabla^d Y_t = (1-L)^dY_t
$$

In [ ]:
transformed = df.copy()
transformed["log"] = np.log(transformed["Robberies"])
transformed["sqrt"] = np.sqrt(transformed["Robberies"])
transformed["diff_1"] = transformed["Robberies"].diff()
transformed["log_diff_1"] = transformed["log"].diff()
transformed["seasonal_diff_12"] = transformed["Robberies"].diff(12)
transformed["diff_1_seasonal_12"] = transformed["Robberies"].diff().diff(12)

transformed.head(15)

In [ ]:
columns_to_plot = ["Robberies", "log", "sqrt", "diff_1", "log_diff_1", "seasonal_diff_12"]

for col in columns_to_plot:
    fig, ax = plt.subplots()
    ax.plot(transformed.index, transformed[col])
    ax.set_title(f"Transformation : {col}")
    ax.set_xlabel("Date")
    ax.set_ylabel(col)
    plt.show()

In [ ]:
for col in ["Robberies", "log", "sqrt", "diff_1", "log_diff_1", "seasonal_diff_12", "diff_1_seasonal_12"]:
    display(stationarity_tests(transformed[col], col))

### Interprétation attendue

Après transformation et différenciation, on cherche une série dont :

- la moyenne est plus stable ;
- la variance est moins dépendante du temps ;
- les tests ADF et KPSS donnent une conclusion favorable à la stationnarité.

Si la différenciation d'ordre 1 rend la série stationnaire, cela suggère un modèle ARIMA avec $d=1$.

## 8. Analyse ACF/PACF

L'autocorrélation au retard $k$ mesure la relation entre $Y_t$ et $Y_{t-k}$ :

$$
\rho(k) = \frac{Cov(Y_t, Y_{t-k})}{Var(Y_t)}
$$

La PACF mesure l'autocorrélation partielle entre $Y_t$ et $Y_{t-k}$ après suppression de l'effet des retards intermédiaires.

Règles pratiques :

- ACF qui coupe rapidement et PACF qui décroît : modèle MA possible ;
- PACF qui coupe rapidement et ACF qui décroît : modèle AR possible ;
- ACF et PACF qui décroissent progressivement : modèle ARMA possible ;
- pics aux multiples de 12 : saisonnalité possible.

In [ ]:
series_for_acf = transformed["diff_1"].dropna()

fig, ax = plt.subplots()
plot_acf(series_for_acf, lags=36, ax=ax)
ax.set_title("ACF de la série différenciée d'ordre 1")
plt.show()

fig, ax = plt.subplots()
plot_pacf(series_for_acf, lags=36, ax=ax, method="ywm")
ax.set_title("PACF de la série différenciée d'ordre 1")
plt.show()

### Hypothèses sur les ordres $p$ et $q$

À partir des graphiques ACF/PACF, on peut proposer plusieurs modèles candidats, par exemple :

- ARIMA(1,1,0) ;
- ARIMA(0,1,1) ;
- ARIMA(1,1,1) ;
- ARIMA(2,1,1) ;
- ARIMA(1,1,2).

Ces choix seront ensuite validés par les performances, AIC/BIC et diagnostic des résidus.

## 9. Découpage chronologique train/test

Pour une série temporelle, il ne faut pas mélanger aléatoirement les données. Le modèle doit être entraîné sur le passé et évalué sur le futur.

Nous utilisons ici environ 80 % des observations pour l'entraînement et 20 % pour le test.

In [ ]:
train_size = int(len(df) * 0.8)
train = df["Robberies"].iloc[:train_size]
test = df["Robberies"].iloc[train_size:]

print("Taille train :", len(train))
print("Taille test  :", len(test))
print("Période train:", train.index.min().date(), "→", train.index.max().date())
print("Période test :", test.index.min().date(), "→", test.index.max().date())

fig, ax = plt.subplots()
ax.plot(train.index, train, label="Train")
ax.plot(test.index, test, label="Test")
ax.set_title("Découpage chronologique train/test")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de vols")
ax.legend()
plt.show()

## 10. Fonctions d'évaluation sans librairies avancées

Nous calculons manuellement les métriques avec `numpy`.

### MSE

$$
MSE = \frac{1}{n}\sum_{t=1}^{n}(Y_t - \hat{Y}_t)^2
$$

### RMSE

$$
RMSE = \sqrt{MSE}
$$

### MAE

$$
MAE = \frac{1}{n}\sum_{t=1}^{n}|Y_t - \hat{Y}_t|
$$

Dans ce projet, ces erreurs s'interprètent en nombre de vols mensuels.

In [ ]:
def mse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, aic=np.nan, bic=np.nan):
    return {
        "Modèle": model_name,
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MSE": mse(y_true, y_pred),
        "AIC": aic,
        "BIC": bic
    }

## 11. Modèles de référence : naïf et moyenne mobile

Avant d'utiliser ARIMA, il faut comparer avec des modèles simples.

### Modèle naïf

La prévision du prochain mois est égale à la dernière valeur observée :

$$
\hat{Y}_{t+1}=Y_t
$$

### Moyenne mobile

La prévision est la moyenne des $k$ dernières observations :

$$
\hat{Y}_{t+1}=\frac{1}{k}\sum_{i=0}^{k-1}Y_{t-i}
$$

Un modèle avancé doit battre ces références pour être réellement utile.

In [ ]:
def naive_forecast(train, test):
    history = list(train.values)
    predictions = []
    for actual in test.values:
        pred = history[-1]
        predictions.append(pred)
        history.append(actual)
    return np.array(predictions)

def moving_average_forecast(train, test, window=3):
    history = list(train.values)
    predictions = []
    for actual in test.values:
        pred = np.mean(history[-window:])
        predictions.append(pred)
        history.append(actual)
    return np.array(predictions)

pred_naive = naive_forecast(train, test)
pred_ma3 = moving_average_forecast(train, test, window=3)
pred_ma12 = moving_average_forecast(train, test, window=12)

baseline_results = [
    evaluate_model(test.values, pred_naive, "Naïf"),
    evaluate_model(test.values, pred_ma3, "Moyenne mobile 3 mois"),
    evaluate_model(test.values, pred_ma12, "Moyenne mobile 12 mois")
]

pd.DataFrame(baseline_results)

In [ ]:
fig, ax = plt.subplots()
ax.plot(test.index, test.values, marker="o", label="Valeurs réelles")
ax.plot(test.index, pred_naive, marker="o", label="Naïf")
ax.plot(test.index, pred_ma3, marker="o", label="Moyenne mobile 3")
ax.plot(test.index, pred_ma12, marker="o", label="Moyenne mobile 12")
ax.set_title("Comparaison des baselines sur le test")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de vols")
ax.legend()
plt.show()

## 12. Walk-forward validation

La validation walk-forward simule une prévision réaliste :

1. entraîner le modèle sur les données disponibles ;
2. prédire la prochaine valeur ;
3. observer la vraie valeur ;
4. l'ajouter à l'historique ;
5. répéter.

Cela réduit le risque de fuite de données, car le futur n'est jamais utilisé pour prédire le passé.

In [ ]:
def walk_forward_arima(train, test, order):
    history = list(train.values)
    predictions = []
    for actual in test.values:
        model = ARIMA(history, order=order)
        fitted = model.fit()
        forecast = fitted.forecast(steps=1)[0]
        predictions.append(forecast)
        history.append(actual)
    return np.array(predictions)

## 13. Théorie : AR, MA et ARMA

### Modèle AR(p)

Un modèle autorégressif AR(p) explique la valeur actuelle par ses valeurs passées :

$$
Y_t = c + \phi_1Y_{t-1}+\phi_2Y_{t-2}+...+\phi_pY_{t-p}+\varepsilon_t
$$

En opérateur de retard :

$$
\Phi(L)Y_t = c + \varepsilon_t
$$

avec :

$$
\Phi(L)=1-\phi_1L-...-\phi_pL^p
$$

La stationnarité exige que les racines du polynôme $\Phi(z)=0$ soient à l'extérieur du cercle unité.

### Modèle MA(q)

Un modèle MA(q) explique la valeur actuelle par les erreurs passées :

$$
Y_t = \mu + \varepsilon_t + \theta_1\varepsilon_{t-1}+...+\theta_q\varepsilon_{t-q}
$$

En opérateur de retard :

$$
Y_t = \mu + \Theta(L)\varepsilon_t
$$

avec :

$$
\Theta(L)=1+\theta_1L+...+\theta_qL^q
$$

L'inversibilité exige que les racines du polynôme $\Theta(z)=0$ soient à l'extérieur du cercle unité.

### Modèle ARMA(p,q)

ARMA combine les deux mécanismes :

$$
\Phi(L)Y_t = c + \Theta(L)\varepsilon_t
$$

## 14. Implémentation from scratch : AR(p) par OLS

Pour un modèle AR(p), on peut écrire :

$$
Y_t = c + \phi_1Y_{t-1}+...+\phi_pY_{t-p}+\varepsilon_t
$$

On construit une matrice de design $X$ contenant une constante et les retards de la série. L'estimation par moindres carrés ordinaires est :

$$
\hat{\beta} = (X'X)^{-1}X'Y
$$

Cette partie montre le mécanisme interne d'estimation d'un modèle AR.

In [ ]:
def fit_ar_ols(series, p):
    y = np.asarray(series, dtype=float)
    X, target = [], []
    for t in range(p, len(y)):
        X.append([1.0] + [y[t-i] for i in range(1, p+1)])
        target.append(y[t])
    X = np.asarray(X)
    target = np.asarray(target)
    beta = np.linalg.inv(X.T @ X) @ X.T @ target
    return beta

def predict_ar_one_step(history, beta):
    p = len(beta) - 1
    x = np.array([1.0] + [history[-i] for i in range(1, p+1)])
    return float(x @ beta)

def walk_forward_ar_ols(train, test, p):
    history = list(train.values)
    predictions = []
    for actual in test.values:
        beta = fit_ar_ols(history, p=p)
        pred = predict_ar_one_step(history, beta)
        predictions.append(pred)
        history.append(actual)
    return np.array(predictions)

# Comme AR exige une série stationnaire, on l'applique à la série différenciée puis on reconstruit les niveaux.
def walk_forward_ar_ols_on_diff(train, test, p):
    history_levels = list(train.values)
    predictions_levels = []

    for actual in test.values:
        diff_history = np.diff(history_levels)
        beta = fit_ar_ols(diff_history, p=p)
        pred_diff = predict_ar_one_step(list(diff_history), beta)
        pred_level = history_levels[-1] + pred_diff
        predictions_levels.append(pred_level)
        history_levels.append(actual)

    return np.array(predictions_levels)

pred_ar1_scratch = walk_forward_ar_ols_on_diff(train, test, p=1)
pred_ar2_scratch = walk_forward_ar_ols_on_diff(train, test, p=2)

pd.DataFrame([
    evaluate_model(test.values, pred_ar1_scratch, "AR(1) from scratch sur diff"),
    evaluate_model(test.values, pred_ar2_scratch, "AR(2) from scratch sur diff")
])

## 15. Implémentation simplifiée from scratch : MA(q) et ARMA(p,q)

L'estimation exacte des modèles MA et ARMA est plus complexe que celle d'un modèle AR, car les erreurs passées ne sont pas directement observées.

Principe simplifié :

1. initialiser les résidus à zéro ;
2. prédire la série avec les paramètres courants ;
3. recalculer les résidus ;
4. chercher les paramètres qui minimisent la somme des carrés des erreurs.

La cellule suivante donne une version pédagogique basée sur une optimisation numérique simple.

In [ ]:
from scipy.optimize import minimize

def arma_sse(params, y, p=1, q=1):
    y = np.asarray(y, dtype=float)
    c = params[0]
    phi = params[1:1+p]
    theta = params[1+p:1+p+q]
    errors = np.zeros(len(y))
    preds = np.zeros(len(y))

    start = max(p, q)
    for t in range(start, len(y)):
        ar_part = sum(phi[i-1] * y[t-i] for i in range(1, p+1))
        ma_part = sum(theta[j-1] * errors[t-j] for j in range(1, q+1))
        preds[t] = c + ar_part + ma_part
        errors[t] = y[t] - preds[t]

    return np.sum(errors[start:] ** 2)

def fit_arma_scratch(series, p=1, q=1):
    y = np.asarray(series, dtype=float)
    initial_params = np.zeros(1 + p + q)
    initial_params[0] = np.mean(y)
    result = minimize(arma_sse, initial_params, args=(y, p, q), method="Nelder-Mead")
    return result.x, result.fun

# Démonstration sur la série différenciée d'entraînement
params_arma11, sse_arma11 = fit_arma_scratch(np.diff(train.values), p=1, q=1)
params_arma11, sse_arma11

### Comparaison avec une librairie

Nous comparons les paramètres estimés de façon simplifiée avec ceux de `statsmodels`. Les valeurs ne seront pas forcément identiques, car `statsmodels` utilise des méthodes plus robustes, notamment le maximum de vraisemblance.

In [ ]:
sm_arma11 = ARIMA(np.diff(train.values), order=(1, 0, 1)).fit()
print(sm_arma11.summary())
print("Paramètres from scratch ARMA(1,1) sur diff :", params_arma11)

## 16. Modèles ARIMA avec statsmodels

Un modèle ARIMA(p,d,q) combine :

- $p$ : nombre de retards autorégressifs ;
- $d$ : nombre de différenciations ;
- $q$ : nombre de retards des erreurs.

Formellement :

$$
\Phi(L)(1-L)^dY_t = c + \Theta(L)\varepsilon_t
$$

Nous testons plusieurs combinaisons et nous comparons les modèles par AIC, BIC, MAE, RMSE et MSE.

Pour garder le notebook rapide à exécuter, les modèles candidats sont évalués ici avec une prévision directe sur l'horizon de test. La fonction de walk-forward est fournie plus haut et peut être utilisée pour une évaluation plus stricte sur un modèle sélectionné.

In [ ]:
arima_orders = [
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 1),
    (1, 1, 2),
    (2, 1, 2),
    (3, 1, 1),
]

arima_results = []
arima_predictions = {}

for order in arima_orders:
    try:
        fitted_on_train = ARIMA(train, order=order).fit()
        preds = fitted_on_train.forecast(steps=len(test))
        name = f"ARIMA{order}"
        arima_predictions[name] = np.asarray(preds)
        arima_results.append(evaluate_model(test.values, preds, name, fitted_on_train.aic, fitted_on_train.bic))
    except Exception as e:
        print(f"Erreur pour ARIMA{order}: {e}")

pd.DataFrame(arima_results).sort_values("RMSE")

## 17. Recherche automatique simple par AIC/BIC

Nous pouvons tester une grille de valeurs pour $p$, $d$ et $q$.

Attention : le meilleur AIC/BIC sur l'entraînement n'est pas toujours le meilleur modèle sur le test. Il faut donc combiner les critères statistiques, les erreurs de prévision et le diagnostic des résidus.

In [ ]:
grid_results = []

# Grille volontairement limitée pour garder le notebook rapide et lisible.
# Vous pouvez l'élargir après validation du notebook.
candidate_orders = [
    (0, 1, 1), (1, 1, 0), (1, 1, 1),
    (2, 1, 1), (1, 1, 2), (2, 1, 2),
    (0, 2, 1), (1, 2, 1), (2, 2, 1)
]

for order in candidate_orders:
    try:
        model = ARIMA(train, order=order)
        fitted = model.fit()
        grid_results.append({
            "order": order,
            "AIC": fitted.aic,
            "BIC": fitted.bic
        })
    except Exception:
        continue

grid_df = pd.DataFrame(grid_results).sort_values("AIC")
grid_df.head(10)

## 18. Variante saisonnière : SARIMA

Si la série présente une structure saisonnière, on peut utiliser SARIMA :

$$
SARIMA(p,d,q)(P,D,Q,s)
$$

Pour une série mensuelle, la période saisonnière est généralement :

$$
s = 12
$$

Nous testons quelques configurations simples.

In [ ]:
sarima_configs = [
    ((1, 1, 1), (0, 0, 0, 12)),
    ((1, 1, 1), (1, 0, 0, 12)),
    ((1, 1, 1), (0, 1, 1, 12)),
    ((2, 1, 1), (1, 0, 0, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
]

sarima_results = []
sarima_predictions = {}

for order, seasonal_order in sarima_configs:
    try:
        model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        fitted = model.fit(disp=False)
        preds = fitted.forecast(steps=len(test))
        name = f"SARIMA{order}x{seasonal_order}"
        sarima_predictions[name] = preds.values
        sarima_results.append(evaluate_model(test.values, preds.values, name, fitted.aic, fitted.bic))
    except Exception as e:
        print(f"Erreur pour SARIMA{order}x{seasonal_order}: {e}")

pd.DataFrame(sarima_results).sort_values("RMSE")

## 19. Tableau comparatif global

Nous regroupons les performances des baselines, des modèles from scratch, des modèles ARIMA et des modèles SARIMA.

In [ ]:
all_results = []
all_results.extend(baseline_results)
all_results.append(evaluate_model(test.values, pred_ar1_scratch, "AR(1) from scratch sur diff"))
all_results.append(evaluate_model(test.values, pred_ar2_scratch, "AR(2) from scratch sur diff"))
all_results.extend(arima_results)
all_results.extend(sarima_results)

results_df = pd.DataFrame(all_results).sort_values("RMSE").reset_index(drop=True)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["Modèle"]
best_model_name

In [ ]:
# Récupération des prédictions du meilleur modèle pour visualisation
prediction_dict = {
    "Naïf": pred_naive,
    "Moyenne mobile 3 mois": pred_ma3,
    "Moyenne mobile 12 mois": pred_ma12,
    "AR(1) from scratch sur diff": pred_ar1_scratch,
    "AR(2) from scratch sur diff": pred_ar2_scratch,
}
prediction_dict.update(arima_predictions)
prediction_dict.update(sarima_predictions)

best_preds = prediction_dict[best_model_name]

fig, ax = plt.subplots()
ax.plot(train.index, train.values, label="Train")
ax.plot(test.index, test.values, marker="o", label="Test réel")
ax.plot(test.index, best_preds, marker="o", label=f"Prévision {best_model_name}")
ax.set_title(f"Prévisions du meilleur modèle : {best_model_name}")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de vols")
ax.legend()
plt.show()

### Interprétation du tableau

À commenter dans le rapport :

- quel modèle obtient la plus faible RMSE ;
- quel modèle obtient la plus faible MAE ;
- si le modèle avancé améliore réellement les baselines ;
- si le gain est important ou faible ;
- si un modèle légèrement moins performant mais plus simple peut être préférable.

## 20. Diagnostic des résidus du modèle retenu

Un bon modèle doit produire des résidus proches d'un bruit blanc :

- moyenne proche de zéro ;
- absence d'autocorrélation ;
- variance relativement stable ;
- distribution raisonnablement symétrique.

Nous ajustons le modèle retenu sur l'ensemble train, puis nous analysons ses résidus.

In [ ]:
def fit_model_from_name(model_name, train):
    if model_name.startswith("ARIMA"):
        order = eval(model_name.replace("ARIMA", ""))
        return ARIMA(train, order=order).fit()
    elif model_name.startswith("SARIMA"):
        # Extraction simple du texte : SARIMA(order)x(seasonal_order)
        left, right = model_name.replace("SARIMA", "").split("x")
        order = eval(left)
        seasonal_order = eval(right)
        return SARIMAX(train, order=order, seasonal_order=seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    else:
        # Si le meilleur est une baseline, on choisit le meilleur ARIMA disponible pour le diagnostic statistique.
        best_arima_row = pd.DataFrame(arima_results).sort_values("RMSE").iloc[0]
        order = eval(best_arima_row["Modèle"].replace("ARIMA", ""))
        print("Le meilleur modèle global est une baseline. Diagnostic effectué sur", best_arima_row["Modèle"])
        return ARIMA(train, order=order).fit()

fitted_final = fit_model_from_name(best_model_name, train)
residuals = pd.Series(fitted_final.resid).dropna()

print(fitted_final.summary())

In [ ]:
fig, ax = plt.subplots()
ax.plot(residuals.index if hasattr(residuals, "index") else range(len(residuals)), residuals)
ax.axhline(0, linestyle="--")
ax.set_title("Résidus du modèle retenu")
ax.set_xlabel("Temps")
ax.set_ylabel("Résidu")
plt.show()

fig, ax = plt.subplots()
ax.hist(residuals, bins=15, edgecolor="black")
ax.set_title("Histogramme des résidus")
ax.set_xlabel("Résidu")
ax.set_ylabel("Fréquence")
plt.show()

fig, ax = plt.subplots()
plot_acf(residuals, lags=30, ax=ax)
ax.set_title("ACF des résidus")
plt.show()

### Test de Ljung-Box

Le test de Ljung-Box vérifie si les résidus sont autocorrélés.

Hypothèses :

$$
H_0 : \text{les résidus ne sont pas autocorrélés}
$$

$$
H_1 : \text{les résidus sont autocorrélés}
$$

Si la p-value est supérieure à 0.05, on ne rejette pas $H_0$, ce qui est favorable.

In [ ]:
ljung = acorr_ljungbox(residuals, lags=[6, 12, 18, 24], return_df=True)
ljung

### Normalité et homoscédasticité

Nous utilisons :

- le test de Shapiro-Wilk pour la normalité ;
- un test ARCH pour vérifier une éventuelle hétéroscédasticité conditionnelle.

Ces tests ne sont pas toujours parfaits sur des séries courtes, donc ils doivent être interprétés avec prudence et en complément des graphiques.

In [ ]:
normality_stat, normality_p = shapiro(residuals)
arch_stat, arch_p, _, _ = het_arch(residuals)

pd.DataFrame({
    "Test": ["Shapiro-Wilk normalité", "ARCH homoscédasticité"],
    "Statistique": [normality_stat, arch_stat],
    "p-value": [normality_p, arch_p],
    "Interprétation à 5%": [
        "Normalité non rejetée" if normality_p > 0.05 else "Normalité rejetée",
        "Homoscédasticité non rejetée" if arch_p > 0.05 else "Hétéroscédasticité possible"
    ]
})

## 21. Sélection et justification du modèle final

Le modèle final doit être choisi en combinant :

- la performance sur le test ;
- les critères AIC/BIC ;
- l'analyse des résidus ;
- la simplicité ;
- l'interprétabilité.

Un modèle très complexe avec un faible gain peut ne pas être préférable à un modèle plus simple.

In [ ]:
print("Modèle retenu selon la RMSE :", best_model_name)
print("Performances associées :")
display(results_df.iloc[[0]])

baseline_rmse = results_df.loc[results_df["Modèle"] == "Naïf", "RMSE"].iloc[0]
best_rmse = results_df.iloc[0]["RMSE"]
gain = (baseline_rmse - best_rmse) / baseline_rmse * 100
print(f"Gain relatif de RMSE par rapport au modèle naïf : {gain:.2f}%")

## 22. Conclusion métier

Dans le contexte des vols à main armée à Boston, l'analyse d'une série temporelle permet de comprendre et de prévoir l'évolution mensuelle du phénomène.

Les principaux points à retenir sont :

1. La série présente une dynamique temporelle claire.
2. La tendance observée rend la série originale probablement non stationnaire.
3. Les tests ADF et KPSS permettent de vérifier formellement la stationnarité.
4. La différenciation est utile pour stabiliser la série.
5. Les baselines sont indispensables pour juger la valeur réelle d'un modèle ARIMA/SARIMA.
6. Le modèle retenu doit être justifié par les performances, les critères statistiques et le diagnostic des résidus.

### Limites

- Le modèle utilise uniquement l'historique des vols.
- Il n'intègre pas de variables externes comme les politiques de sécurité, le contexte économique, la population ou les événements exceptionnels.
- Les prévisions peuvent être sensibles aux changements structurels.

### Améliorations possibles

- Tester plus de modèles SARIMA.
- Utiliser une validation walk-forward plus complète pour SARIMA.
- Ajouter des variables explicatives avec SARIMAX.
- Comparer avec d'autres modèles de séries temporelles comme Prophet, ETS ou modèles de machine learning.

## 23. Résumé prêt pour le rapport

La série mensuelle des vols à main armée à Boston a été analysée selon une démarche complète de séries temporelles. L'analyse exploratoire a permis d'identifier une dynamique croissante et une possible non-stationnarité. Les tests ADF et KPSS ont été utilisés pour confirmer la stationnarité ou la non-stationnarité de la série. Des transformations, notamment la différenciation, ont ensuite été appliquées pour stabiliser la série.

Des modèles de référence, comme le modèle naïf et la moyenne mobile, ont d'abord été construits afin de disposer d'un point de comparaison. Ensuite, plusieurs modèles ARIMA et SARIMA ont été testés. Les modèles ont été comparés avec MAE, RMSE, MSE, AIC et BIC. Le modèle final a été sélectionné en tenant compte à la fois de la performance prédictive, de la simplicité et du diagnostic des résidus.

Cette démarche montre que la modélisation ARIMA peut capturer une partie importante de la dynamique des vols mensuels, mais qu'elle reste limitée par l'absence de facteurs explicatifs externes.